In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression

def predict_xgboost(
    print_type_Printed,
    Binder_Weight,
    SP, 
    VMA, 
    Glass_fiber,            
    PP_fiber, 
    Mix_Weight, 
    day_01, 
    sand_water_ratio, 
    target_day):
    # Load the model
    model = joblib.load("../models/xgboost-istonic-enforced-regression-model.pkl")
    
    # Prepare input data as a DataFrame to match training format
    inputs = pd.DataFrame([[
        print_type_Printed, Binder_Weight, SP, VMA, Glass_fiber, 
        PP_fiber, Mix_Weight, day_01, sand_water_ratio
    ]], columns=['print_type_Printed', 'Binder_Weight', 'SP', 'VMA', 
                 'Glass_fiber', 'PP_fiber', 'Mix_Weight', 'day_01', 'Sand/Water'])
    
    # Get raw predictions for all 30 days
    y_pred = model.predict(inputs)
    
    x_iso = np.arange(y_pred.shape[1])
    y_pred_mono = IsotonicRegression(increasing=False).fit_transform(x_iso, y_pred[0])
    
    # Return the value for the specific day (1-indexed)
    return y_pred_mono[target_day - 1]

In [2]:
def predict_xgboost_delta(print_type_Printed, Binder_Weight, SP, VMA, Glass_fiber, 
                        PP_fiber, Mix_Weight, day_01, sand_water_ratio, target_day):
    # Load model
    model = joblib.load("../models/xgboost-delta-model.pkl")
    
    # Prepare input
    inputs = pd.DataFrame([[
        print_type_Printed, Binder_Weight, SP, VMA, Glass_fiber, 
        PP_fiber, Mix_Weight, day_01, sand_water_ratio
    ]], columns=['print_type_Printed', 'Binder_Weight', 'SP', 'VMA', 
                 'Glass_fiber', 'PP_fiber', 'Mix_Weight', 'day_01', 'Sand/Water'])
    
    # Predict deltas (changes)
    delta_pred = model.predict(inputs)
    
    # Post-processing: Reconstruct the sequence
    # y_pred = [day_01, day_01 - delta1, day_01 - (delta1 + delta2)...]
    y0 = np.array([day_01])
    y_rest = y0 - np.cumsum(delta_pred, axis=1)
    y_full_sequence = np.hstack([y0, y_rest[0]])
    
    return y_full_sequence[target_day - 1]

In [3]:
import numpy as np
import pandas as pd
import joblib

def predict_elastic_net(
    print_type_Cast,
    print_type_Printed,
    Nano_Silica,
    Fly_ash,
    Ultrafine_Fly_ash,
    Ordinary_Portland_Cement,
    Binder_Weight,
    SP,
    VMA,
    Glass_fiber,
    PP_fiber,
    Sand,
    Water,
    Mix_Weight,
    day_01,
    Sand_Water,
    target_day
):
    # Load trained Elastic Net model
    model = joblib.load("../models/elasticnet-delta-model.pkl")

    # Prepare input with EXACT feature names and order
    inputs = pd.DataFrame([[
        print_type_Cast,
        print_type_Printed,
        Nano_Silica,
        Fly_ash,
        Ultrafine_Fly_ash,
        Ordinary_Portland_Cement,
        Binder_Weight,
        SP,
        VMA,
        Glass_fiber,
        PP_fiber,
        Sand,
        Water,
        Mix_Weight,
        day_01,
        Sand_Water
    ]], columns=[
        'print_type_Cast',
        'print_type_Printed',
        'Nano-Silica',
        'Fly_ash',
        'Ultrafine_Fly_ash',
        'Ordinary_Portland_Cement',
        'Binder_Weight',
        'SP',
        'VMA',
        'Glass_fiber',
        'PP_fiber',
        'Sand',
        'Water',
        'Mix_Weight',
        'day_01',
        'Sand/Water'
    ])

    # Predict deltas (Δ strength per day)
    delta_pred = model.predict(inputs)

    # Reconstruct full strength sequence
    y0 = np.array([day_01])
    y_rest = y0 - np.cumsum(delta_pred, axis=1)
    y_full_sequence = np.hstack([y0, y_rest[0]])

    return y_full_sequence[target_day - 1]


In [4]:
predicted_value_1 = predict_xgboost(
    print_type_Printed=1,     # 1 = Printed, 0 = Cast
    Binder_Weight=777.7,       # kg
    SP=0.7777,                   # superplasticizer
    VMA=2.72195,                 # viscosity modifying agent
    Glass_fiber=3.8885,          # %
    PP_fiber=0.0,             # %
    Mix_Weight=2223.83815,         # kg
    day_01=5.701,              # Day-1 strength
    sand_water_ratio=34.285636,     # Sand / Water
    target_day=2             # Day to predict (1–30)
)


In [5]:
predicted_value_2 = predict_xgboost_delta(
    print_type_Printed=1,     # 1 = Printed, 0 = Cast
    Binder_Weight=777.7,       # kg
    SP=0.7777,                   # superplasticizer
    VMA=2.72195,                 # viscosity modifying agent
    Glass_fiber=3.8885,          # %
    PP_fiber=0.0,             # %
    Mix_Weight=2223.83815,         # kg
    day_01=5.701,              # Day-1 strength
    sand_water_ratio=34.285636,     # Sand / Water
    target_day=2             # Day to predict (1–30)
)


In [6]:
predicted_value_3 = predict_elastic_net(
    print_type_Cast=0,            # 1 = Cast, 0 = Printed
    print_type_Printed=1,         # 1 = Printed, 0 = Cast
    Nano_Silica=15.55,             # kg
    Fly_ash=155.54,                # kg
    Ultrafine_Fly_ash=101.1,       # kg
    Ordinary_Portland_Cement=505.51, # kg
    Binder_Weight=777.7,          # kg
    SP=0.7777,                    # superplasticizer (% or kg as used in training)
    VMA=2.72195,                  # viscosity modifying agent
    Glass_fiber=3.8885,           # %
    PP_fiber=0.0,                 # %
    Sand=1100.0,                  # kg
    Water=320.0,                  # kg
    Mix_Weight=2223.83815,        # kg
    day_01=5.701,                 # Day-1 strength
    Sand_Water=34.285636,         # Sand / Water ratio
    target_day=2                  # Day to predict (1–30)
)


In [7]:
print(f"predicted values from model 1: {predicted_value_1}")
print(f"predicted values from model 2: {predicted_value_2}")
print(f"predicted values from model 3: {predicted_value_3}")

predicted values from model 1: 5.550037384033203
predicted values from model 2: 5.557560185551643
predicted values from model 3: 5.566249363554891
